In [4]:
import pandas as pd
import numpy as np
from datetime import timedelta
import random
import string

np.random.seed(42)

# -------------------------
# SETTINGS
# -------------------------
start_date = "2022-01-01"
end_date = "2024-12-31"
dates = pd.date_range(start_date, end_date, freq="D")

# -------------------------
# DRUG LIST (Realistic)
# -------------------------
drugs = [
    {"drug_id": "D001", "drug_name": "Artemether-Lumefantrine", "category": "Antimalarial"},
    {"drug_id": "D002", "drug_name": "Amoxicillin 500mg", "category": "Antibiotic"},
    {"drug_id": "D003", "drug_name": "Paracetamol 500mg", "category": "Analgesic"},
    {"drug_id": "D004", "drug_name": "Ceftriaxone Injection", "category": "Antibiotic"},
    {"drug_id": "D005", "drug_name": "Metformin 850mg", "category": "Antidiabetic"},
    {"drug_id": "D006", "drug_name": "Salbutamol Inhaler", "category": "Respiratory"},
    {"drug_id": "D007", "drug_name": "ORS Sachets", "category": "Rehydration"},
    {"drug_id": "D008", "drug_name": "Coartem Pediatric", "category": "Antimalarial"},
    {"drug_id": "D009", "drug_name": "Azithromycin 250mg", "category": "Antibiotic"},
    {"drug_id": "D010", "drug_name": "Ibuprofen 400mg", "category": "Analgesic"},
]

sales_records = []
stock_records = []
opening_stock_records = []

transaction_id = 1
stock_id = 1

# -------------------------
# SIMULATION
# -------------------------
for drug in drugs:

    drug_id = drug["drug_id"]
    drug_name = drug["drug_name"]
    category = drug["category"]

    base = np.random.randint(20, 120)
    stock = np.random.randint(2000, 5000)
    # Capture opening stock at initialization for this drug
    opening_stock_records.append([
        drug_id,
        drug_name,
        stock,
    ])
    next_restock = dates[0] + timedelta(days=np.random.randint(20, 40))

    for date in dates:

        # -------------------------
        # TREND (small growth yearly)
        # -------------------------
        trend = (date.year - 2022) * 2

        # -------------------------
        # SEASONALITY (drug-specific)
        # -------------------------
        month = date.month
        seasonal = 0

        # Antimalarials peak in rainy seasons
        if category == "Antimalarial" and month in [3,4,5,10,11]:
            seasonal = base * 0.5

        # Respiratory drugs peak mid-year
        elif category == "Respiratory" and month in [6,7,8]:
            seasonal = base * 0.4

        # Rehydration salts peak hot months
        elif category == "Rehydration" and month in [1,2]:
            seasonal = base * 0.3

        # Antibiotics mild seasonal fluctuation
        elif category == "Antibiotic" and month in [6,7]:
            seasonal = base * 0.2

        # -------------------------
        # RANDOM NOISE
        # -------------------------
        noise = np.random.normal(0, 5)

        daily_demand = max(0, int(base + trend + seasonal + noise))

        # -------------------------
        # SALES (if stock available)
        # -------------------------
        if stock > 0:
            quantity = min(daily_demand, stock)
            stock -= quantity

            remaining = quantity

            while remaining > 0:
                q = min(np.random.randint(1, 6), remaining)
                sales_records.append([
                    f"T{transaction_id}",
                    date,
                    drug_id,
                    drug_name,
                    category,
                    q,
                    round(np.random.uniform(5, 50), 2),
                    "F001"
                ])
                transaction_id += 1
                remaining -= q

        # -------------------------
        # RESTOCKING
        # -------------------------
        if date >= next_restock:
            qty_received = np.random.randint(800, 3000)
            stock += qty_received

            stock_records.append([
                f"S{stock_id}",
                drug_id,
                drug_name,
                date,
                qty_received,
                np.random.randint(0,7),
                ''.join(random.choices(string.ascii_uppercase + string.digits, k=6)),
                date + timedelta(days=np.random.randint(365, 730))
            ])

            stock_id += 1
            next_restock = date + timedelta(days=np.random.randint(20, 45))

# -------------------------
# CREATE DATAFRAMES
# -------------------------
sales_df = pd.DataFrame(sales_records, columns=[
    "transaction_id", "transaction_date", "drug_id",
    "drug_name", "category",
    "quantity_dispensed", "unit_price", "facility_id"
])

stock_df = pd.DataFrame(stock_records, columns=[
    "stock_id", "drug_id", "drug_name",
    "stock_received_date", "quantity_received",
    "delivery_delay_days", "batch_number", "expiry_date"
])

opening_stock_df = pd.DataFrame(opening_stock_records, columns=[
    "drug_id", "drug_name", "opening_stock_units"
])

# -------------------------
# SAVE FILES
# -------------------------
#sales_df.to_csv("sales_transactions.csv", index=False)
#stock_df.to_csv("stock_receipts.csv", index=False)
opening_stock_df.to_csv("opening_stock.csv", index=False)

print("Dataset generated successfully!")
print("Sales shape:", sales_df.shape)
print("Stock shape:", stock_df.shape)
print("Opening stock shape:", opening_stock_df.shape)

Dataset generated successfully!
Sales shape: (223389, 8)
Stock shape: (340, 8)
Opening stock shape: (10, 3)


## Opening stock export

This notebook now also exports `opening_stock.csv` in the `data/` folder.

For each drug we capture the **initial stock level** at simulation start (`opening_stock_units`).
This allows downstream notebooks and the dashboard to compute a more realistic inventory balance:

\[
\text{current\_stock} = \text{opening\_stock\_units} + \text{total\_received} - \text{total\_dispensed}
\]

The stock receipts and sales datasets remain unchanged.